# Human preference classification — Gemma 2 2B + LoRA

**Experiment 1:** correct the inputs and train adapters, not just the classification head.
Based on your `main_reviewed(2).ipynb`. This is a **local training/evaluation notebook**, not a Kaggle hidden-test submission notebook.

Restart the kernel, close other kernels holding a model on your GPU, then run top to bottom.
Use the same project directory containing `data/train.csv` and `data/test.csv`.
Your existing Hugging Face access to Gemma and working CUDA PyTorch installation are reused.

Default run: **one epoch, BF16 LoRA, 1,536-token maximum, batch 2 × accumulation 16**.
There is **no 4-bit quantization, augmentation, calibration, or ensemble** in this first experiment.
A fresh prompt-grouped validation split is saved; its score is **not directly comparable** with the old random split.

# Setup
## Dependencies and environment
The first cell installs only missing helper packages; it does not explicitly reinstall or upgrade PyTorch.
Run this notebook in the same CUDA-enabled environment as your previous notebook.

In [1]:
import importlib.util
import subprocess
import sys

required = {
    "peft": "peft>=0.17.0",
    "accelerate": "accelerate>=1.0.0",
    "datasets": "datasets>=3.0.0",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
}
missing = [package for module, package in required.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

In [2]:
import gc
import hashlib
import inspect
import json
import math
import os
import time
from datetime import datetime
from importlib.metadata import version
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import DatasetDict, load_dataset
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from scipy.special import softmax
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import StratifiedGroupKFold
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    DataCollatorWithPadding, Trainer, TrainingArguments, set_seed,
)
from IPython.display import display

if not torch.cuda.is_available():
    raise RuntimeError("Select your CUDA-enabled PyTorch kernel before running this notebook.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("BF16 is not available. Check the CUDA PyTorch installation for your RTX 5090.")

versions = {p: version(p) for p in
            ["torch", "transformers", "peft", "accelerate", "datasets", "scikit-learn"]}
print(json.dumps(versions, indent=2))
print("Python:", sys.version.split()[0])
print("GPU:", torch.cuda.get_device_name(0))
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")

c:\Users\yana\Desktop\Library\AI\ai-competitions\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1011 06:48:36.774000 14948 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


{
  "torch": "2.12.1+cu130",
  "transformers": "5.19.0",
  "peft": "0.21.2",
  "accelerate": "1.15.0",
  "datasets": "5.1.0",
  "scikit-learn": "1.9.1"
}
Python: 3.14.5
GPU: NVIDIA GeForce RTX 5090
GPU memory: 31.8 GiB


## Parameters
Keep these defaults for the first run. For an out-of-memory error, restart the kernel and set
`TRAIN_BATCH_SIZE=1`, `GRAD_ACCUMULATION=32`, and `EVAL_BATCH_SIZE=1`.
Each run gets a new output directory so it cannot overwrite your head-only checkpoints.

In [3]:
SEED = 42
MODEL_NAME = "google/gemma-2-2b-it"
DATA_DIR = Path("./data")
COMPETITION = "llm-classification-finetuning"
RUN_DIR = Path("../tmp/outputs") / f"arena_lora_2b_{datetime.now():%Y%m%d_%H%M%S}"
RUN_DIR.mkdir(parents=True, exist_ok=False)

MAX_LENGTH = 1536
PROMPT_BUDGET = 256
NUM_EPOCHS = 1
LEARNING_RATE = 5e-5
TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 4
GRAD_ACCUMULATION = 16
LORA_RANK = 16
LORA_ALPHA = 32

LABEL_NAMES = ["win_a", "win_b", "tie"]
LABEL_COLUMNS = ["winner_model_a", "winner_model_b", "winner_tie"]
ID2LABEL = dict(enumerate(LABEL_NAMES))
LABEL2ID = {name: i for i, name in ID2LABEL.items()}
DEVICE = torch.device("cuda:0")
set_seed(SEED)
print("Output:", RUN_DIR.resolve())
print("Effective batch size:", TRAIN_BATCH_SIZE * GRAD_ACCUMULATION)

Output: C:\Users\yana\Desktop\Library\AI\ai-competitions\tmp\outputs\arena_lora_2b_20261011_064837
Effective batch size: 32


# 1. Data
## Load and validate labels
Class IDs remain **0 = A wins, 1 = B wins, 2 = tie**. This is single-label, three-class classification.
The download step is skipped when both CSV files already exist. If downloading is needed, your Kaggle credentials and accepted competition rules are required.

In [4]:
if not all((DATA_DIR / name).is_file() for name in ["train.csv", "test.csv"]):
    import kagglehub
    kagglehub.competition_download(handle=COMPETITION, output_dir=str(DATA_DIR))

train_raw = load_dataset("csv", data_files=str(DATA_DIR / "train.csv"))["train"]
test_raw = load_dataset("csv", data_files=str(DATA_DIR / "test.csv"))["train"]

one_hot = np.column_stack([train_raw[col] for col in LABEL_COLUMNS])
assert np.isin(one_hot, [0, 1]).all(), "Labels must be binary indicator columns."
assert (one_hot.sum(axis=1) == 1).all(), "Each row must have exactly one winning class."
assert len(set(train_raw["id"])) == len(train_raw), "Duplicate training IDs found."
assert len(set(test_raw["id"])) == len(test_raw), "Duplicate test IDs found."

features = ["id", "prompt", "response_a", "response_b"]
train_raw = train_raw.select_columns(features).add_column("labels", one_hot.argmax(axis=1))
test_raw = test_raw.select_columns(features)

## Parse JSON and preserve turn numbering
Each CSV text field is a JSON list of turns, not a plain sentence.
Text, Markdown, code indentation, and empty turns are preserved; missing text is marked explicitly.
A group key is made from the complete normalized prompt conversation **only for splitting**; the model's text is not whitespace-normalized.

In [5]:
def parse_turns(value):
    """Decode one competition field. Never evaluate Python code from the data."""
    turns = json.loads(value) if isinstance(value, str) else value
    if turns is None:
        return [""]
    if not isinstance(turns, list):
        raise ValueError("Expected a JSON list of turns.")
    if not all(turn is None or isinstance(turn, str) for turn in turns):
        raise ValueError("Turns must be strings or null.")
    return ["" if turn is None else turn for turn in turns] or [""]


def format_turns(turns):
    return "\n\n".join(
        f"[Turn {i + 1}]\n{turn if turn else '[No text]'}"
        for i, turn in enumerate(turns)
    )


def prepare_row(row):
    try:
        p, a, b = [parse_turns(row[col]) for col in ["prompt", "response_a", "response_b"]]
    except (ValueError, TypeError) as exc:
        raise ValueError(f"Invalid conversation in row id={row['id']}: {exc}") from exc
    # Group identical normalized full prompt conversations, not individual response pairs.
    normalized = json.dumps([" ".join(turn.split()) for turn in p], ensure_ascii=False)
    group = hashlib.sha256(normalized.encode("utf-8")).hexdigest()
    n_turns = max(len(p), len(a), len(b))
    mismatched = len({len(p), len(a), len(b)}) > 1
    p, a, b = [turns + [""] * (n_turns - len(turns)) for turns in [p, a, b]]
    return {
        "prompt_text": format_turns(p),
        "response_a_text": format_turns(a),
        "response_b_text": format_turns(b),
        "prompt_group": group,
        "num_turns": n_turns,
        "mismatched_turns": mismatched,
    }

raw_text_columns = ["prompt", "response_a", "response_b"]
train_prepared = train_raw.map(prepare_row, remove_columns=raw_text_columns,
                               desc="Parse training conversations")
test_prepared = test_raw.map(prepare_row, remove_columns=raw_text_columns,
                             desc="Parse test conversations")

## Fixed validation split and constant baselines
Use the first fold of a seeded, five-way stratified **group** split (approximately 80/20).
All rows with the same normalized full prompt conversation stay on one side.
This guards exact normalized prompt overlap, not every possible near-duplicate.
Splits and IDs are saved for reproducible comparisons in subsequent experiments.

In [6]:
y_all = np.asarray(train_prepared["labels"], dtype=np.int64)
groups = np.asarray(train_prepared["prompt_group"])
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, valid_idx = next(splitter.split(np.zeros(len(y_all)), y_all, groups))
assert not set(groups[train_idx]).intersection(groups[valid_idx])

ds = DatasetDict({
    "train": train_prepared.select(train_idx),
    "valid": train_prepared.select(valid_idx),
    "test": test_prepared,
})
y_train = np.asarray(ds["train"]["labels"], dtype=np.int64)
y_valid = np.asarray(ds["valid"]["labels"], dtype=np.int64)
assert set(y_train) == {0, 1, 2} and set(y_valid) == {0, 1, 2}

manifest = pd.DataFrame({"id": train_prepared["id"], "split": "train"})
manifest.loc[valid_idx, "split"] = "valid"
manifest.to_csv(RUN_DIR / "split.csv", index=False)

prior = np.bincount(y_train, minlength=3) / len(y_train)
baseline_scores = {
    "uniform_log_loss": float(log_loss(y_valid, np.full((len(y_valid), 3), 1 / 3), labels=[0, 1, 2])),
    "class_prior_log_loss": float(log_loss(y_valid, np.tile(prior, (len(y_valid), 1)), labels=[0, 1, 2])),
}
print({name: len(data) for name, data in ds.items()})
print("Normalized prompt overlap: 0")
print("Rows with mismatched turn counts:", sum(train_prepared["mismatched_turns"]))
display(pd.DataFrame({"train": prior, "valid": np.bincount(y_valid, minlength=3) / len(y_valid)},
                     index=LABEL_NAMES))
print(json.dumps(baseline_scores, indent=2))

{'train': 45981, 'valid': 11496, 'test': 3}
Normalized prompt overlap: 0
Rows with mismatched turn counts: 0


,train,valid
win_a,0.349079,0.349078
win_b,0.341924,0.341858
tie,0.308997,0.309064


{
  "uniform_log_loss": 1.0986122886681096,
  "class_prior_log_loss": 1.0972323373988446
}


# 2. LoRA baseline
## Tokenizer and actual per-example token budgets
The three sections contain the same turn numbering. They are encoded separately, then assembled from token IDs.
The prompt gets up to 256 tokens; the remaining space is shared by A and B, including all headers and special tokens in the total limit.
Unused answer space is given to the other answer. An answer cannot consume the other answer's allocation.

For a truncated section, keep its beginning and end, with a visible omission marker.
**This still discards some text and possibly middle turns**; it does not preserve an unlimited conversation.
The audit below reports truncation and writes a decoded example showing exactly what reaches the model.

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "right"
assert tokenizer.bos_token_id is not None and tokenizer.eos_token_id is not None
assert tokenizer.pad_token_id is not None


def encode_text(text):
    return tokenizer.encode(text, add_special_tokens=False)


prefix_ids = encode_text("Choose which response a human would prefer: A, B, or tie.\n\n[Prompt]\n")
a_header_ids = encode_text("\n\n[Response A]\n")
b_header_ids = encode_text("\n\n[Response B]\n")
suffix_ids = encode_text("\n\n[Decision: A, B, or tie]")
omission_ids = encode_text("\n[... omitted ...]\n")
overhead = len(prefix_ids) + len(a_header_ids) + len(b_header_ids) + len(suffix_ids) + 2
assert MAX_LENGTH % 8 == 0 and MAX_LENGTH <= tokenizer.model_max_length
assert MAX_LENGTH - overhead - PROMPT_BUDGET > 2 * len(omission_ids)


def trim_tokens(ids, budget):
    """A token budget for ONE example, with head-and-tail preservation."""
    if len(ids) <= budget:
        return ids
    if budget <= len(omission_ids):
        return ids[:budget]
    remaining = budget - len(omission_ids)
    head = (remaining + 1) // 2
    tail = remaining // 2
    return ids[:head] + omission_ids + (ids[-tail:] if tail else [])


def assemble_input(p, a, b):
    p_budget = min(len(p), PROMPT_BUDGET)
    answer_space = MAX_LENGTH - overhead - p_budget
    a_budget = min(len(a), answer_space // 2)
    b_budget = min(len(b), answer_space // 2)
    # Redistribute unused answer space without ever dropping the other answer's allocation.
    spare = answer_space - a_budget - b_budget
    extra_a = min(spare, len(a) - a_budget)
    a_budget += extra_a
    b_budget += min(spare - extra_a, len(b) - b_budget)
    input_ids = (
        [tokenizer.bos_token_id] + prefix_ids + trim_tokens(p, p_budget)
        + a_header_ids + trim_tokens(a, a_budget)
        + b_header_ids + trim_tokens(b, b_budget)
        + suffix_ids + [tokenizer.eos_token_id]
    )
    assert len(input_ids) <= MAX_LENGTH
    return input_ids, [len(p) > p_budget, len(a) > a_budget, len(b) > b_budget]

In [8]:
def tokenize(batch):
    # Each of these is a batch of complete example strings; never slice the row list as a token limit.
    p_batch = tokenizer(batch["prompt_text"], add_special_tokens=False, truncation=False)["input_ids"]
    a_batch = tokenizer(batch["response_a_text"], add_special_tokens=False, truncation=False)["input_ids"]
    b_batch = tokenizer(batch["response_b_text"], add_special_tokens=False, truncation=False)["input_ids"]
    result = {key: [] for key in ["input_ids", "attention_mask", "length", "original_length",
                                  "truncated_prompt", "truncated_a", "truncated_b"]}
    for p, a, b in zip(p_batch, a_batch, b_batch):
        ids, flags = assemble_input(p, a, b)
        result["input_ids"].append(ids)
        result["attention_mask"].append([1] * len(ids))
        result["length"].append(len(ids))
        result["original_length"].append(overhead + len(p) + len(a) + len(b))
        for name, flag in zip(["truncated_prompt", "truncated_a", "truncated_b"], flags):
            result[name].append(flag)
    if "labels" in batch:
        result["labels"] = batch["labels"]
    return result


tokenized = DatasetDict({
    name: data.map(tokenize, batched=True, batch_size=128, remove_columns=data.column_names,
                   desc=f"Tokenize {name}")
    for name, data in ds.items()
})

audit_rows = []
for name, data in tokenized.items():
    lengths = np.asarray(data["length"])
    audit_rows.append({
        "split": name, "rows": len(data), "mean_tokens": float(lengths.mean()),
        "max_tokens": int(lengths.max()),
        **{col + "_fraction": float(np.mean(data[col]))
           for col in ["truncated_prompt", "truncated_a", "truncated_b"]},
    })
audit = pd.DataFrame(audit_rows)
audit.to_csv(RUN_DIR / "tokenization_audit.csv", index=False)
display(audit)

example_index = int(np.argmax(tokenized["train"]["original_length"]))
decoded_example = tokenizer.decode(tokenized["train"][example_index]["input_ids"], skip_special_tokens=False)
(RUN_DIR / "decoded_example.txt").write_text(decoded_example, encoding="utf-8")
print("Longest-original training example after tokenization:\n")
print(decoded_example)

# Keep only genuine model inputs. Audit columns stay available in `tokenized`.
train_tokenized = tokenized["train"].select_columns(["input_ids", "attention_mask", "labels"])
valid_tokenized = tokenized["valid"].select_columns(["input_ids", "attention_mask", "labels"])
test_tokenized = tokenized["test"].select_columns(["input_ids", "attention_mask"])
collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

,split,rows,mean_tokens,max_tokens,truncated_prompt_fraction,truncated_a_fraction,truncated_b_fraction
0,train,45981,661.728779,1536,0.073704,0.055305,0.055414
1,valid,11496,651.721642,1536,0.068372,0.054280,0.052975
2,test,3,714.333333,1536,0.000000,0.333333,0.333333


Longest-original training example after tokenization:

<bos>Choose which response a human would prefer: A, B, or tie.

[Prompt]
[Turn 1]
Let's play a text based adventure game, it's a fantasy setting and mostly focused on dialog. You will play the characters I interact with. Let's start with a sassy, sarcastic companion that isn't afraid to banter with me and stop me if they deem it necessary.

[Turn 2]
I tell her: My new best friend, you say I'll be the judge of that. Tell me why I should tell you anything about myself and we'll go from there.

[Turn 3]
I tell her: Touché, nice
[... omitted ...]
 is as red as that of any villain, it always starts like this. I turn back to the guardian and the bandits and say: With this, my job is done, and I bid you farewell. I say as I turn to leave.

[Turn 35]
I tell her: Peacemaker you say, a lot of people would disagree, but I like it. Today went a lot better than I hoped it would, but don't think that is the norm, as you will find out. Let's trav

## Model: adapters plus classification head
Use ordinary BF16 LoRA, **not QLoRA**. The original backbone weights are frozen by PEFT; adapters and `score` are trainable and saved.
Do not run the old head-only freezing loop afterward.

`eager` attention keeps Gemma's attention softcapping and avoids a FlashAttention/Triton installation requirement.
It is a correctness/compatibility choice, not the fastest possible attention backend.
Gradient checkpointing trades extra computation for lower activation memory.

In [9]:
def load_backbone(revision=None):
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        revision=revision,
        num_labels=3,
        problem_type="single_label_classification",
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        pad_token_id=tokenizer.pad_token_id,
        dtype=torch.bfloat16,
        attn_implementation="eager",
    )
    model.config.use_cache = False
    model.score.float()
    return model


set_seed(SEED)
base_model = load_backbone()
model_revision = getattr(base_model.config, "_commit_hash", None)
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    target_modules="all-linear",
    modules_to_save=["score"],
    bias="none",
)
base_model = get_peft_model(base_model, lora_config)
# Keep optimizer-updated parameters in FP32 while the frozen backbone remains BF16.
for param in base_model.parameters():
    if param.requires_grad:
        param.data = param.data.float()
base_model.to(DEVICE)
base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
base_model.print_trainable_parameters()

trainable_names = [name for name, param in base_model.named_parameters() if param.requires_grad]
assert any("lora_" in name for name in trainable_names), "LoRA adapters are not trainable."
assert any("score" in name for name in trainable_names), "Classification head is not trainable."
assert all("lora_" in name or "score" in name for name in trainable_names)
trainable_count = sum(p.numel() for p in base_model.parameters() if p.requires_grad)

Loading weights: 100%|██████████| 288/288 [00:01<00:00, 152.67it/s]
[transformers] Gemma2ForSequenceClassification LOAD REPORT from: google/gemma-2-2b-it
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 20,773,632 || all params: 2,635,122,432 || trainable%: 0.7883


## Loss and metrics
The explicit cross-entropy callback divides each microbatch's summed loss by the number of examples in the **whole accumulated batch**.
It is passed through Trainer's public `compute_loss_func` interface; there is no manual patch of Trainer internals.
During ordinary evaluation, it uses the current batch size instead.
The small synthetic test checks both the loss and its gradient, including an incomplete microbatch.

In [10]:
def compute_loss(outputs, labels, num_items_in_batch=None):
    loss_sum = F.cross_entropy(outputs.logits.float(), labels.long(), reduction="sum")
    denominator = labels.numel() if num_items_in_batch is None else num_items_in_batch
    return loss_sum / denominator


def get_logits(predictions):
    return predictions[0] if isinstance(predictions, tuple) else predictions


def compute_metrics(eval_pred):
    logits = np.asarray(get_logits(eval_pred.predictions), dtype=np.float64)
    probabilities = softmax(logits, axis=-1)
    if not np.isfinite(probabilities).all():
        raise ValueError("Non-finite validation probabilities detected.")
    return {
        "log_loss": float(log_loss(eval_pred.label_ids, probabilities, labels=[0, 1, 2])),
        "accuracy": float(accuracy_score(eval_pred.label_ids, probabilities.argmax(axis=-1))),
    }

In [11]:
from types import SimpleNamespace

check_logits = torch.tensor([[1., 0., -1.], [0., 2., 1.], [1., 1., 2.],
                             [-1., 2., 0.], [3., 1., 0.]], requires_grad=True)
check_labels = torch.tensor([0, 1, 2, 0, 2])
full_loss = F.cross_entropy(check_logits, check_labels)
accumulated_loss = sum(
    compute_loss(SimpleNamespace(logits=check_logits[start:start + 2]),
                 check_labels[start:start + 2], num_items_in_batch=len(check_labels))
    for start in range(0, len(check_labels), 2)
)
torch.testing.assert_close(full_loss, accumulated_loss)
full_gradient = torch.autograd.grad(full_loss, check_logits, retain_graph=True)[0]
accumulated_gradient = torch.autograd.grad(accumulated_loss, check_logits)[0]
torch.testing.assert_close(full_gradient, accumulated_gradient)
print("Cross-entropy accumulation check passed.")
del check_logits, check_labels, full_loss, accumulated_loss, full_gradient, accumulated_gradient

Cross-entropy accumulation check passed.


## GPU forward/backward smoke test
Test a maximum-length physical training batch **before** starting the full epoch.
This verifies finite loss and nonzero gradients in both the adapters and the head.
No optimizer step is taken, so the initial model weights are unchanged.

In [12]:
probe_indices = np.argsort(np.asarray(tokenized["train"]["length"]))[-TRAIN_BATCH_SIZE:]
probe = collator([train_tokenized[int(i)] for i in probe_indices])
probe = {key: value.to(DEVICE) for key, value in probe.items()}
base_model.train()
base_model.zero_grad(set_to_none=True)
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
probe_start = time.perf_counter()

with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    probe_output = base_model(input_ids=probe["input_ids"], attention_mask=probe["attention_mask"])
    probe_loss = compute_loss(probe_output, probe["labels"])
assert torch.isfinite(probe_loss), "Non-finite smoke-test loss."
probe_loss.backward()

grad_norms = {"adapter": 0.0, "head": 0.0}
for name, param in base_model.named_parameters():
    if param.requires_grad and param.grad is not None:
        assert torch.isfinite(param.grad).all(), f"Non-finite gradient in {name}"
        key = "adapter" if "lora_" in name else "head"
        grad_norms[key] += float(param.grad.float().norm().item())
assert grad_norms["adapter"] > 0 and grad_norms["head"] > 0, grad_norms

torch.cuda.synchronize()
smoke_test = {
    "loss": float(probe_loss.item()),
    "seconds": time.perf_counter() - probe_start,
    "peak_allocated_gib": torch.cuda.max_memory_allocated() / 2**30,
    "gradient_norm_sums": grad_norms,
}
print(json.dumps(smoke_test, indent=2))
base_model.zero_grad(set_to_none=True)
del probe, probe_output, probe_loss
# The loop variables can hold the final parameter, but no graph or extra model copy.
gc.collect()
torch.cuda.empty_cache()
set_seed(SEED)

{
  "loss": 1.121019721031189,
  "seconds": 1.200657500000034,
  "peak_allocated_gib": 6.471183776855469,
  "gradient_norm_sums": {
    "adapter": 392.8138997554779,
    "head": 77.75634002685547
  }
}


## Training
One epoch first. Both checkpointing and evaluation happen at the end of each epoch, with **lowest log loss** selecting the saved model.
Early stopping is omitted because it is not useful for a one-epoch run.
An integer warmup-step count avoids differences between Transformers' older and newer warmup APIs.

In [13]:
steps_per_epoch = math.ceil(math.ceil(len(train_tokenized) / TRAIN_BATCH_SIZE) / GRAD_ACCUMULATION)
warmup_steps = max(1, int(0.05 * steps_per_epoch * NUM_EPOCHS))

# The length-grouping argument was renamed in newer Transformers releases.
argument_names = inspect.signature(TrainingArguments).parameters
length_grouping = ({"train_sampling_strategy": "group_by_length"}
                   if "train_sampling_strategy" in argument_names else {"group_by_length": True})

training_args = TrainingArguments(
    output_dir=str(RUN_DIR / "checkpoints"),
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="linear",
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    max_grad_norm=1.0,
    optim="adamw_torch",
    bf16=True,
    fp16=False,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="log_loss",
    greater_is_better=False,
    logging_steps=25,
    logging_first_step=True,
    report_to="none",
    label_names=["labels"],
    dataloader_num_workers=0,  # Safe default for Windows notebooks.
    dataloader_pin_memory=True,
    seed=SEED,
    data_seed=SEED,
    disable_tqdm=False,
    **length_grouping,
)

run_config = {
    "model_name": MODEL_NAME, "model_revision": model_revision,
    "method": "BF16 LoRA; no quantization", "seed": SEED,
    "max_length": MAX_LENGTH, "prompt_budget": PROMPT_BUDGET,
    "epochs": NUM_EPOCHS, "learning_rate": LEARNING_RATE,
    "train_batch_size": TRAIN_BATCH_SIZE, "eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUMULATION,
    "lora_rank": LORA_RANK, "lora_alpha": LORA_ALPHA, "lora_dropout": 0.05,
    "attention_backend": "eager", "label_names": LABEL_NAMES,
    "split": "StratifiedGroupKFold, 5 folds, first held-out fold, full prompt groups",
    "train_rows": len(ds["train"]), "valid_rows": len(ds["valid"]),
    "trainable_parameters": trainable_count, "versions": versions,
    "python": sys.version, "gpu": torch.cuda.get_device_name(0),
}
(RUN_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2), encoding="utf-8")

934

In [14]:
trainer = Trainer(
    model=base_model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=valid_tokenized,
    data_collator=collator,
    compute_loss_func=compute_loss,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

torch.cuda.reset_peak_memory_stats()
train_result = trainer.train()
train_metrics = dict(train_result.metrics)
training_peak_gib = torch.cuda.max_memory_allocated() / 2**30
best_checkpoint_log_loss = float(trainer.state.best_metric)
log_history = pd.DataFrame(trainer.state.log_history)
log_history.to_csv(RUN_DIR / "training_log.csv", index=False)

ADAPTER_DIR = RUN_DIR / "adapter"
trainer.save_model(str(ADAPTER_DIR))  # Saves LoRA adapters AND the trained classification head.
tokenizer.save_pretrained(str(ADAPTER_DIR))
trainer.save_state()
print("Saved trained adapter and head:", ADAPTER_DIR.resolve())

Epoch,Training Loss,Validation Loss,Log Loss,Accuracy,Runtime,Samples Per Second,Steps Per Second
1,0.978037,0.964659,0.964659,0.528358,423.531600,27.143000,6.786000


Saved trained adapter and head: C:\Users\yana\Desktop\Library\AI\ai-competitions\tmp\outputs\arena_lora_2b_20261011_064837\adapter


# 3. Reload and evaluate the saved model
Release the training model, then reload the **base model + saved adapter + saved head** before producing predictions.
This avoids accidentally evaluating a new random head or forgetting the adapter in inference.
The extra validation pass collects per-example probabilities and logits for our next review.

In [15]:
del trainer, base_model
# Do not keep a second full model on the GPU while reloading.
gc.collect()
torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_DIR))
collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)
base_model = load_backbone(revision=model_revision)
base_model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR), is_trainable=False)
base_model.config.use_cache = False
base_model.eval()

inference_args = TrainingArguments(
    output_dir=str(RUN_DIR / "inference"),
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    bf16=True,
    report_to="none",
    label_names=["labels"],
    dataloader_num_workers=0,
    disable_tqdm=False,
)
inference_trainer = Trainer(
    model=base_model,
    args=inference_args,
    data_collator=collator,
    compute_loss_func=compute_loss,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)
valid_output = inference_trainer.predict(valid_tokenized, metric_key_prefix="valid")
valid_logits = np.asarray(get_logits(valid_output.predictions), dtype=np.float32)
valid_probs = softmax(valid_logits.astype(np.float64), axis=-1)
assert valid_logits.shape == (len(y_valid), 3)
assert np.array_equal(valid_output.label_ids, y_valid), "Validation row order changed."
assert np.isfinite(valid_probs).all() and np.allclose(valid_probs.sum(axis=1), 1.0)

Loading weights: 100%|██████████| 288/288 [00:02<00:00, 123.56it/s]
[transformers] Gemma2ForSequenceClassification LOAD REPORT from: google/gemma-2-2b-it
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
valid_predictions = pd.DataFrame({
    "id": ds["valid"]["id"], "label": y_valid,
    "predicted_label": valid_probs.argmax(axis=1),
    "num_turns": ds["valid"]["num_turns"],
    "length": tokenized["valid"]["length"],
    **{col: tokenized["valid"][col] for col in ["truncated_prompt", "truncated_a", "truncated_b"]},
})
for i, column in enumerate(LABEL_COLUMNS):
    valid_predictions[column] = valid_probs[:, i]
valid_predictions["row_log_loss"] = -np.log(np.clip(valid_probs[np.arange(len(y_valid)), y_valid],
                                                    np.finfo(np.float64).eps, 1.0))
valid_predictions.to_csv(RUN_DIR / "valid_predictions.csv", index=False)
np.savez_compressed(RUN_DIR / "valid_logits.npz", logits=valid_logits, labels=y_valid,
                    ids=np.asarray(ds["valid"]["id"]))

summary = {
    "run_directory": str(RUN_DIR.resolve()),
    "config": run_config,
    "baseline_scores": baseline_scores,
    "validation": {key: float(value) for key, value in valid_output.metrics.items()},
    "best_checkpoint_log_loss": best_checkpoint_log_loss,
    "reload_log_loss_difference": float(valid_output.metrics["valid_log_loss"] - best_checkpoint_log_loss),
    "beats_uniform": bool(valid_output.metrics["valid_log_loss"] < baseline_scores["uniform_log_loss"]),
    "beats_class_prior": bool(valid_output.metrics["valid_log_loss"] < baseline_scores["class_prior_log_loss"]),
    "train_metrics": train_metrics,
    "training_peak_allocated_gib": training_peak_gib,
    "smoke_test": smoke_test,
    "tokenization_audit": audit.to_dict(orient="records"),
}
(RUN_DIR / "run_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

comparison = pd.DataFrame({
    "model": ["Uniform", "Train class priors", "Reloaded Gemma 2 2B + LoRA"],
    "validation_log_loss": [baseline_scores["uniform_log_loss"], baseline_scores["class_prior_log_loss"],
                            valid_output.metrics["valid_log_loss"]],
})
display(comparison)
print(f"Validation accuracy: {valid_output.metrics['valid_accuracy']:.4f}")
print(f"Reload log-loss difference: {summary['reload_log_loss_difference']:+.6f}")
if abs(summary["reload_log_loss_difference"]) > 0.01:
    print("WARNING: saved-model evaluation differs materially; inspect the reload before submitting.")
print("Summary to share:", (RUN_DIR / "run_summary.json").resolve())

,model,validation_log_loss
0,Uniform,1.098612
1,Train class priors,1.097232
2,Reloaded Gemma 2 2B + LoRA,0.964771


Validation accuracy: 0.5277
Reload log-loss difference: +0.000112
Summary to share: C:\Users\yana\Desktop\Library\AI\ai-competitions\tmp\outputs\arena_lora_2b_20261011_064837\run_summary.json


## Local test predictions
This uses exactly the same preprocessing and reloaded model.
The downloaded test file may contain only three demonstration rows: predictions for those rows are **not** a complete hidden-test submission.
Kaggle scoring needs a separate offline inference notebook with these adapters, the base weights, and this same tokenizer/input construction available locally.

In [17]:
test_output = inference_trainer.predict(test_tokenized)
test_logits = np.asarray(get_logits(test_output.predictions), dtype=np.float64)
test_probs = softmax(test_logits, axis=-1)
assert test_probs.shape == (len(ds["test"]), 3)
assert np.isfinite(test_probs).all() and np.allclose(test_probs.sum(axis=1), 1.0)

test_predictions = pd.DataFrame(test_probs, columns=LABEL_COLUMNS)
test_predictions.insert(0, "id", ds["test"]["id"])
test_predictions.to_csv(RUN_DIR / "test_predictions.csv", index=False)
display(test_predictions.head())
print(f"Wrote predictions for {len(test_predictions):,} local test rows only.")
print("Finished:", RUN_DIR.resolve())

,id,winner_model_a,winner_model_b,winner_tie
0,136060,0.208516,0.431202,0.360283
1,211333,0.334367,0.381859,0.283774
2,1233961,0.137514,0.503005,0.359481


Wrote predictions for 3 local test rows only.
Finished: C:\Users\yana\Desktop\Library\AI\ai-competitions\tmp\outputs\arena_lora_2b_20261011_064837
